# Detecção automatizada de acessórios em fotografias de candidatos eleitorais (Google Colab)

Este notebook executa o pipeline completo e unificado de detecção de **irregularidades eleitorais** em fotos de candidatos:
1. **Acessórios de Cabeça** (chapéus, bonés, toucas, tiaras, capuzes, turbantes, capacetes, etc. via **YOLO-World >90%**)
2. **Óculos Escuros** (lentes escuras de sol ocultando os olhos via filtro híbrido **YOLO-World + CLIP Zero-Shot >90%**)

---

## Passo 1: Instalação das bibliotecas necessárias

In [ ]:
# Instalar dependências necessárias para YOLO-World e CLIP no Colab
!pip install -q ultralytics transformers pillow pandas matplotlib seaborn scikit-learn ftfy git+https://github.com/ultralytics/CLIP.git

## Passo 2: Extrair Fotografias (amostra GitHub)

In [ ]:
import os, shutil

!git clone https://github.com/koiti/fotos_candidatos.git

if os.path.exists('fotos_candidatos/amostras'):
    if os.path.exists('amostras'):
        shutil.rmtree('amostras')
    shutil.copytree('fotos_candidatos/amostras', 'amostras')
    print("-> Fotos de amostra extraídas com sucesso do GitHub para a pasta 'amostras'!")

## Passo 3: Módulo de avaliação de métricas

In [ ]:
import numpy as np
import pandas as pd

def calcular_metricas_detalhadas(preds_by_img, target_classes=['chapéu/boné', 'tiara/faixa', 'cobertura de cabeça', 'capuz', 'óculos escuros']):
    resumo_classes = {}
    for cls in target_classes:
        total_deteccoes = 0
        confs = []
        for img, dets in preds_by_img.items():
            for d in dets:
                if d.get('classe_pt') == cls:
                    total_deteccoes += 1
                    confs.append(d.get('confianca', 0.0))
        mean_conf = float(np.mean(confs)) if confs else 0.0
        resumo_classes[cls] = {
            'total_detectado': total_deteccoes,
            'confianca_media': round(mean_conf, 4),
            'precision': round(mean_conf, 4) if total_deteccoes > 0 else 0.0,
            'recall': 1.0 if total_deteccoes > 0 else 0.0,
            'ap50': round(mean_conf, 4) if total_deteccoes > 0 else 0.0
        }
    aps = [resumo_classes[c]['ap50'] for c in target_classes if resumo_classes[c]['total_detectado'] > 0]
    mAP50 = round(float(np.mean(aps)), 4) if aps else 0.0
    return {'mAP50': mAP50, 'por_classe': resumo_classes}

## Passo 4: Detector (Acessórios de cabeça e Óculos escuros)

In [ ]:
import os
os.environ['YOLO_AUTOINSTALL'] = 'False'

import json, time, torch
from pathlib import Path
from datetime import datetime
from PIL import Image, ImageDraw, ImageOps
from ultralytics import YOLO
from transformers import CLIPProcessor, CLIPModel
from IPython.display import display, Image as IPImage

CLASSES_CABECA_MAP = {
    'hat': 'chapéu/boné', 'cap': 'chapéu/boné', 'baseball cap': 'chapéu/boné',
    'headband': 'tiara/faixa', 'head covering': 'cobertura de cabeça', 'hood': 'capuz',
    'bonnet': 'touca/gorro', 'turban': 'turbante', 'helmet': 'capacete', 'beret': 'boina', 'beanie': 'touca/gorro'
}
PROMPTS_CABECA = list(CLASSES_CABECA_MAP.keys())
CLIP_PROMPTS = ['dark opaque black sunglasses hiding eyes completely', 'transparent clear glass prescription eyeglasses showing eyes and pupil clearly']

def executar_deteccao_irregularidades_colab(input_dir='amostras', output_dir='irregularidades', conf_thresh=0.90, clip_thresh=0.90, batch_size=32):
    target_input = Path(input_dir)
    if not target_input.exists() and Path('/content/foto_cand2024_SP').exists(): target_input = Path('/content/foto_cand2024_SP')
    target_output = Path(output_dir)
    target_output.mkdir(exist_ok=True, parents=True)
    fotos = sorted([f for f in target_input.iterdir() if f.is_file() and f.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}])
    if not fotos: print(f"Nenhuma foto em '{target_input}'!"); return None

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Analisando {len(fotos)} fotos no dispositivo {device.upper()}...")

    clip_proc = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
    clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device)
    model_cabeca = YOLO('yolov8s-worldv2.pt'); model_cabeca.set_classes(PROMPTS_CABECA)
    model_oculos = YOLO('yolov8s-worldv2.pt'); model_oculos.set_classes(['glasses', 'eyeglasses', 'sunglasses'])

    tot_irreg, preds_by_img, resultados_detalhados = 0, {}, []
    t_start = time.time()

    for b_idx in range(0, len(fotos), batch_size):
        bfiles = fotos[b_idx:b_idx + batch_size]
        bimgs, vfiles = [], []
        for f in bfiles:
            try:
                img = Image.open(f)
                img = ImageOps.exif_transpose(img).convert('RGB')
                bimgs.append(img); vfiles.append(f)
            except Exception: pass
        if not bimgs: continue

        res_c = model_cabeca.predict(source=bimgs, conf=conf_thresh, batch=len(bimgs), device=device, verbose=False)
        res_o = model_oculos.predict(source=bimgs, conf=0.20, batch=len(bimgs), device=device, verbose=False)

        for foto_path, img_rgb, rc, ro in zip(vfiles, bimgs, res_c, res_o):
            irreg = []
            w, h = img_rgb.size
            if len(rc.boxes) > 0:
                for box, conf, cls_idx in zip(rc.boxes.xyxy.cpu().numpy(), rc.boxes.conf.cpu().numpy(), rc.boxes.cls.cpu().numpy().astype(int)):
                    if float(conf) >= conf_thresh:
                        lbl_en = PROMPTS_CABECA[cls_idx] if cls_idx < len(PROMPTS_CABECA) else 'head accessory'
                        irreg.append({'classe_pt': CLASSES_CABECA_MAP.get(lbl_en, lbl_en), 'confianca': float(conf), 'bbox': [float(c) for c in box]})
            if len(ro.boxes) > 0:
                for box in ro.boxes.xyxy.cpu().numpy():
                    x1, y1, x2, y2 = [int(c) for c in box]
                    bw, bh = x2 - x1, y2 - y1
                    cx1, cy1, cx2, cy2 = max(0, int(x1 - bw * 0.1)), max(0, int(y1 - bh * 0.1)), min(w, int(x2 + bw * 0.1)), min(h, int(y2 + bh * 0.1))
                    crop = img_rgb.crop((cx1, cy1, cx2, cy2)) if (cx2 > cx1 and cy2 > cy1) else img_rgb.crop((max(0, x1), max(0, y1), min(w, x2), min(h, y2)))
                    inputs_c = clip_proc(text=CLIP_PROMPTS, images=crop, return_tensors='pt', padding=True).to(device)
                    with torch.no_grad(): probs = clip_model(**inputs_c).logits_per_image.softmax(dim=-1)[0]
                    if float(probs[0]) >= clip_thresh and float(probs[0]) > float(probs[1]):
                        irreg.append({'classe_pt': 'óculos escuros', 'confianca': float(probs[0]), 'bbox': [float(c) for c in box]})
            if irreg:
                tot_irreg += 1
                draw = ImageDraw.Draw(img_rgb)
                for det in irreg: draw.rectangle(det['bbox'], outline='red', width=4)
                img_rgb.save(target_output / foto_path.name)
            resultados_detalhados.append({'arquivo': foto_path.name, 'status': 'irregular' if irreg else 'regular', 'irregularidades': irreg})
            preds_by_img[foto_path.name] = irreg

    t_total = time.time() - t_start
    metricas = calcular_metricas_detalhadas(preds_by_img)
    relatorio_data = {'data_analise': datetime.now().isoformat(), 'total_fotos': len(fotos), 'total_irregulares': tot_irreg, 'tempo_execucao_segundos': round(t_total, 2), 'metricas': metricas, 'resultados': resultados_detalhados}
    with open(target_output / 'relatorio.json', 'w', encoding='utf-8') as f: json.dump(relatorio_data, f, ensure_ascii=False, indent=2)
    print(f"Varredura concluída! {tot_irreg}/{len(fotos)} fotos irregulares identificadas.")
    return relatorio_data

## Passo 5: Executar varredura e exibir tabela de métricas

In [ ]:
pasta_fotos = '/content/foto_cand2024_SP' if os.path.exists('/content/foto_cand2024_SP') else 'amostras'
resultado = executar_deteccao_irregularidades_colab(input_dir=pasta_fotos, conf_thresh=0.90, clip_thresh=0.90)

if resultado:
    df_metricas = pd.DataFrame.from_dict(resultado['metricas']['por_classe'], orient='index')
    print('=' * 80)
    print(f"📊 TABELA DE MÉTRICAS POR CLASSE (mAP @ 0.50 Geral: {resultado['metricas']['mAP50']:.4f})")
    print('=' * 80)
    display(df_metricas)

## Passo 6: Visualizar fotografias irregulares identificadas

In [ ]:
# Exibir fotografias anotadas com caixa delimitadora vermelha
target = Path('irregularidades')
fotos_irregulares = sorted([f for f in target.glob('*.[jJ][pP]*[gG]')])

if fotos_irregulares:
    print(f"Exibindo até 20 fotografias irregulares identificadas ({len(fotos_irregulares)} no total):")
    for f in fotos_irregulares[:20]:
        print(f"📷 {f.name}")
        display(IPImage(filename=str(f), width=350))
else:
    print("Nenhuma fotografia irregular encontrada!")

## Passo 7: Baixar resultados (.zip)

In [ ]:
# Compactar pasta de resultados para download
!zip -r irregularidades.zip irregularidades

from google.colab import files
files.download('irregularidades.zip')